# 07 - Wrapper Feature Selection

This notebook performs the fourth classification experiment using a
wrapper-based feature-selection strategy.

Unlike filter methods, wrapper methods evaluate feature subsets according to
the predictive performance of a classification algorithm.

In this experiment, Sequential Feature Selection is used with a Decision Tree
as the estimator responsible for selecting the feature subset.

The selected subset is then evaluated using the same set of classifiers,
cross-validation folds, preprocessing strategy and evaluation metrics used in
the previous experiments.

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.feature_selection import SequentialFeatureSelector

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

pd.set_option("display.max_columns", None)

## 1. Dataset Preparation

The same dataset preparation used in the previous experiments is maintained.

Identifier and constant attributes are removed, while `Attrition` is encoded
as a binary target:

- `No` -> 0
- `Yes` -> 1

Keeping the dataset preparation unchanged allows the effect of the wrapper
strategy to be evaluated under the same experimental conditions.

In [2]:
DATA_PATH = "../../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv"

df = pd.read_csv(DATA_PATH)

columns_to_remove = [
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

df_model = df.drop(columns=columns_to_remove)

X = df_model.drop(columns=["Attrition"])

y = df_model["Attrition"].map({
    "No": 0,
    "Yes": 1
})

print("Predictor matrix shape:", X.shape)
print("Target shape:", y.shape)

Predictor matrix shape: (1470, 30)
Target shape: (1470,)


## 2. Preprocessing Definition

The same preprocessing strategy used in the previous experiments is preserved.

Numerical predictors are standardized and categorical predictors are one-hot
encoded.

The preprocessing transformer is included inside the complete pipeline so
that all transformations are fitted exclusively using the training data of
each cross-validation fold.

In [3]:
numerical_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numerical predictors:", len(numerical_features))
print("Categorical predictors:", len(categorical_features))

Numerical predictors: 23
Categorical predictors: 7


In [4]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

## 3. Wrapper Feature Selection

Wrapper methods select features according to the predictive performance
obtained by a classifier.

Sequential Forward Selection is used in this experiment.

The procedure starts with an empty feature set and iteratively adds the feature
that provides the largest improvement in classification performance.

A Decision Tree is used as the estimator responsible for evaluating candidate
feature subsets.

The wrapper retains 15 transformed features, matching the dimensionality used
in the univariate and multivariate experiments.

In [5]:
N_SELECTED_FEATURES = 15

print(
    f"Number of transformed features to retain: "
    f"{N_SELECTED_FEATURES}"
)

Number of transformed features to retain: 15


## 4. Wrapper Estimator

A Decision Tree is used as the estimator inside the Sequential Feature
Selector.

The estimator is kept simple and deterministic by fixing its random state.

The subset selected by this wrapper is later evaluated with all classifiers
used in the project.

In [6]:
wrapper_estimator = DecisionTreeClassifier(
    random_state=42
)

## 5. Evaluation Strategy

The same stratified 10-fold cross-validation procedure used in the previous
experiments is maintained.

For each outer cross-validation fold:

1. Preprocessing is fitted using only the training data.
2. Sequential Feature Selection is applied using only the training data.
3. Fifteen transformed features are selected.
4. The classifier is trained using the selected features.
5. The learned preprocessing and selected subset are applied to the
   validation fold.
6. Performance is measured exclusively on the validation fold.

This prevents validation information from influencing feature selection.

In [7]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

## 6. Evaluation Metrics

The same evaluation metrics used in the previous experiments are retained:

- Accuracy
- Precision
- Recall
- F1-score
- Balanced Accuracy

Particular attention is given to recall, F1-score, and balanced accuracy due
to the imbalance of the target variable.

In [8]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "balanced_accuracy": "balanced_accuracy"
}

## 7. Classification Algorithms

The same classifier configurations used in the previous experiments are
maintained.

This ensures that the feature-selection strategy remains the main
experimental factor that changes.

In [9]:
models = {
    "K-NN": KNeighborsClassifier(),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "SVM": SVC(),

    "Neural Network": MLPClassifier(
        random_state=42,
        max_iter=2000
    )
}

## 8. Evaluation Function

For each classifier, preprocessing, wrapper feature selection, and
classification are combined into a single scikit-learn `Pipeline`.

Sequential Feature Selection is performed independently within each training
fold, preventing information leakage.

In [19]:
def evaluate_model(
    name,
    classifier,
    X,
    y,
    cv,
    scoring
):
    """
    Evaluate a classifier using preprocessing,
    wrapper feature selection, and stratified
    cross-validation.
    """

    feature_selector = SequentialFeatureSelector(
        estimator=wrapper_estimator,
        n_features_to_select=N_SELECTED_FEATURES,
        direction="forward",
        scoring="f1",
        cv=5,
        n_jobs=-1
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessing", preprocessor),
            (
                "feature_selection",
                feature_selector
            ),
            ("classifier", classifier)
        ]
    )

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    result = {
        "Model": name,
        "Selected Features": N_SELECTED_FEATURES
    }

    for metric in scoring.keys():

        metric_values = scores[f"test_{metric}"]

        result[f"{metric}_mean"] = metric_values.mean()
        result[f"{metric}_std"] = metric_values.std()

    return result

## 9. Model Evaluation

Each classifier is evaluated after wrapper-based feature selection.

Because Sequential Feature Selection evaluates many candidate subsets, this
experiment is computationally more expensive than the filter-based
experiments.

In [11]:
results = []

for name, model in models.items():

    print(f"Evaluating {name}...")

    result = evaluate_model(
        name=name,
        classifier=model,
        X=X,
        y=y,
        cv=cv,
        scoring=scoring
    )

    results.append(result)

print("\nEvaluation completed.")

Evaluating K-NN...
Evaluating Decision Tree...
Evaluating SVM...
Evaluating Neural Network...

Evaluation completed.


## 10. Complete Cross-Validation Results

The following table reports the mean and standard deviation of each evaluation
metric across the ten outer cross-validation folds.

In [12]:
wrapper_results = pd.DataFrame(results)

display(
    wrapper_results.round(4)
)

,Model,Selected Features,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,balanced_accuracy_mean,balanced_accuracy_std
0,K-NN,15,0.8395,0.0236,0.5043,0.1630,0.2364,0.1061,0.3146,0.1235,0.5959,0.0536
1,Decision Tree,15,0.7782,0.0248,0.3358,0.0701,0.3879,0.0896,0.3594,0.0774,0.6205,0.0495
2,SVM,15,0.8456,0.0106,0.6638,0.2409,0.0971,0.0377,0.1672,0.0607,0.5433,0.0198
3,Neural Network,15,0.8401,0.0266,0.5278,0.1362,0.3165,0.0507,0.3919,0.0709,0.6286,0.0320


## 11. Wrapper Selection Performance Summary

The following table summarizes the mean predictive performance obtained after
wrapper-based feature selection.

In [13]:
summary_results = wrapper_results[
    [
        "Model",
        "Selected Features",
        "accuracy_mean",
        "precision_mean",
        "recall_mean",
        "f1_mean",
        "balanced_accuracy_mean"
    ]
].copy()

summary_results.columns = [
    "Model",
    "Selected Features",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Balanced Accuracy"
]

summary_results = summary_results.sort_values(
    "F1",
    ascending=False
)

display(
    summary_results.round(4)
)

,Model,Selected Features,Accuracy,Precision,Recall,F1,Balanced Accuracy
3,Neural Network,15,0.8401,0.5278,0.3165,0.3919,0.6286
1,Decision Tree,15,0.7782,0.3358,0.3879,0.3594,0.6205
0,K-NN,15,0.8395,0.5043,0.2364,0.3146,0.5959
2,SVM,15,0.8456,0.6638,0.0971,0.1672,0.5433


## 12. Inspection of the Selected Features

After completing the honest nested cross-validation experiment, preprocessing
and Sequential Feature Selection are fitted to the complete dataset solely for
interpretation.

This additional fit is not used to estimate predictive performance.

Its purpose is to identify the transformed features selected by the wrapper
when all available data are considered.

In [ ]:
inspection_selector = SequentialFeatureSelector(
    estimator=wrapper_estimator,
    n_features_to_select=N_SELECTED_FEATURES,
    direction="forward",
    scoring="f1",
    cv=5,
    n_jobs=-1
)

inspection_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        (
            "feature_selection",
            inspection_selector
        )
    ]
)

inspection_pipeline.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('feature_selection', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](30,)","['Age','BusinessTravel','DailyRate',...,'YearsInCurrentRole', 'YearsSinceLastPromotion','YearsWithCurrManager']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,30
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be

In [15]:
fitted_preprocessor = (
    inspection_pipeline
    .named_steps["preprocessing"]
)

fitted_selector = (
    inspection_pipeline
    .named_steps["feature_selection"]
)

all_feature_names = (
    fitted_preprocessor
    .get_feature_names_out()
)

selected_mask = (
    fitted_selector
    .get_support()
)

selected_feature_names = (
    all_feature_names[selected_mask]
)

print(
    f"Selected features: "
    f"{len(selected_feature_names)}"
)

display(
    pd.Series(
        selected_feature_names,
        name="Selected transformed feature"
    )
)

Selected features: 15


0                                  num__Age
1                            num__DailyRate
2                       num__JobInvolvement
3                        num__MonthlyIncome
4                   num__NumCompaniesWorked
5             num__RelationshipSatisfaction
6                     num__StockOptionLevel
7                    num__TotalWorkingYears
8     cat__BusinessTravel_Travel_Frequently
9                      cat__JobRole_Manager
10             cat__JobRole_Sales Executive
11               cat__MaritalStatus_Married
12                cat__MaritalStatus_Single
13                         cat__OverTime_No
14                        cat__OverTime_Yes
Name: Selected transformed feature, dtype: str

In [16]:
wrapper_feature_selection = pd.DataFrame({
    "Feature": all_feature_names,
    "Selected": selected_mask
})

wrapper_feature_selection = (
    wrapper_feature_selection
    .sort_values(
        "Selected",
        ascending=False
    )
)

display(
    wrapper_feature_selection.head(20)
)

,Feature,Selected
0,num__Age,True
14,num__RelationshipSatisfaction,True
49,cat__OverTime_No,True
48,cat__MaritalStatus_Single,True
47,cat__MaritalStatus_Married,True
44,cat__JobRole_Sales Executive,True
40,cat__JobRole_Manager,True
1,num__DailyRate,True
24,cat__BusinessTravel_Travel_Frequently,True
16,num__TotalWorkingYears,True


## 13. Export Results

The wrapper cross-validation results, summary table, and selected feature
information are stored for the final comparison between feature-selection
strategies.

In [17]:
wrapper_results.to_csv(
    "../../results/pa1/wrapper_results.csv",
    index=False
)

summary_results.to_csv(
    "../../results/pa1/wrapper_summary.csv",
    index=False
)

wrapper_feature_selection.to_csv(
    "../../results/pa1/wrapper_selected_features.csv",
    index=False
)

print(
    "Wrapper feature-selection results saved successfully."
)

Wrapper feature-selection results saved successfully.


## 14. Comparison with Previous Experiments

The wrapper results are compared with the baseline, univariate filter, and
multivariate filter experiments.

All experiments use the same dataset, classifiers, outer cross-validation
folds, preprocessing strategy, and evaluation metrics.

In [18]:
baseline_summary = pd.read_csv(
    "../../results/pa1/baseline_summary.csv"
)

univariate_summary = pd.read_csv(
    "../../results/pa1/univariate_summary.csv"
)

multivariate_summary = pd.read_csv(
    "../../results/pa1/multivariate_summary.csv"
)

baseline_comparison = baseline_summary.copy()
baseline_comparison["Experiment"] = "All Features"

univariate_comparison = univariate_summary.copy()
univariate_comparison["Experiment"] = "Univariate"

multivariate_comparison = multivariate_summary.copy()
multivariate_comparison["Experiment"] = "Multivariate"

wrapper_comparison = summary_results.copy()
wrapper_comparison["Experiment"] = "Wrapper"

comparison = pd.concat(
    [
        baseline_comparison,
        univariate_comparison,
        multivariate_comparison,
        wrapper_comparison
    ],
    ignore_index=True
)

comparison = comparison[
    [
        "Experiment",
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Balanced Accuracy"
    ]
]

display(
    comparison.round(4)
)

,Experiment,Model,Accuracy,Precision,Recall,F1,Balanced Accuracy
0,All Features,Neural Network,0.8490,0.5492,0.4219,0.4738,0.6765
1,All Features,Decision Tree,0.7925,0.3604,0.3670,0.3620,0.6206
2,All Features,SVM,0.8653,0.8363,0.2241,0.3461,0.6064
3,All Features,K-NN,0.8442,0.5412,0.1350,0.2115,0.5577
4,Univariate,Decision Tree,0.7789,0.3546,0.4219,0.3809,0.6347
5,Univariate,Neural Network,0.8163,0.4167,0.3293,0.3642,0.6197
6,Univariate,K-NN,0.8463,0.5761,0.2062,0.2968,0.5877
7,Univariate,SVM,0.8571,0.7933,0.1692,0.2737,0.5793
8,Multivariate,Neural Network,0.8245,0.4501,0.3534,0.3902,0.6341
9,Multivariate,Decision Tree,0.7898,0.3652,0.3958,0.3748,0.6306


## 15. Main Observations

The wrapper-based experiment reduces the transformed representation from 51
features to 15 features selected according to the predictive performance of a
Decision Tree.

The results show that the effect of wrapper selection depends considerably on
the classifier:

- The **Neural Network** achieves the strongest performance after wrapper
  selection, with an F1-score of 0.3919 and a balanced accuracy of 0.6286.
  However, this remains clearly below its baseline performance using all
  features (F1 = 0.4738; balanced accuracy = 0.6765).

- The **Decision Tree** obtains an F1-score of 0.3594 and a balanced accuracy
  of 0.6205. Its performance is slightly lower than its baseline result and
  also lower than the results obtained with the univariate and multivariate
  filters.

- **K-NN benefits from wrapper selection compared with the original feature
  representation.** Its F1-score increases from 0.2115 in the baseline to
  0.3146, while recall increases from 0.1350 to 0.2364. Among the three
  feature-selection strategies, the wrapper provides the best K-NN result.

- The **SVM** retains relatively high accuracy (0.8456) and precision
  (0.6638), but its recall falls to 0.0971. Consequently, its F1-score is only
  0.1672. This again illustrates why accuracy alone is insufficient for this
  imbalanced classification problem.

- Compared with the two filter approaches, wrapper selection produces the
  highest F1-score for K-NN and a similar Neural Network performance to the
  multivariate filter. However, it does not improve the best overall results
  obtained using the complete feature representation.

- The feature subset selected for interpretation includes variables related
  to age, income, job involvement, professional experience, business travel,
  job role, marital status, and overtime. Several of these characteristics
  were also identified by the previous feature-selection strategies and
  exploratory analysis.

Overall, wrapper-based feature selection substantially reduces the number of
features and improves K-NN performance, but it does not outperform the best
baseline model. The Neural Network using all transformed features remains the
strongest configuration according to F1-score and balanced accuracy.